# generator-loss-fool-discriminator — worked example 3: Generator and discriminator loss on the same fakes

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `generator-loss-fool-discriminator`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

The adversarial signal is the *opposite labels* the two players assign to the identical fake batch: the discriminator scores fakes against target 0 ('these are fake'), the generator scores them against target 1 ('I want these called real'). Both are BCE; only the target flips.

## Worked solution

We compute both losses from one set of D predictions on fakes to expose the asymmetry.

1. `d_pred_fake` is D's probability on the fakes, shape `(B,)`.
2. The discriminator's contribution from the fake batch uses target 0: `F.binary_cross_entropy(d_pred_fake, t.zeros_like(d_pred_fake))` — it is penalized for calling fakes real.
3. The generator's loss uses target 1: `F.binary_cross_entropy(d_pred_fake, t.ones_like(d_pred_fake))` — it is penalized for D calling fakes fake.
4. We return both. When D is confident the batch is fake (prediction near 0), the D-loss is small but the G-loss is large, and vice versa — a perfect tug-of-war. We confirm this opposite-direction behavior on a near-zero prediction batch.

In [ ]:
import torch as t
import torch.nn.functional as F

t.manual_seed(2)

def both_losses(d_pred_fake):
    d_loss = F.binary_cross_entropy(d_pred_fake, t.zeros_like(d_pred_fake))
    g_loss = F.binary_cross_entropy(d_pred_fake, t.ones_like(d_pred_fake))
    return d_loss, g_loss

pred = t.full((8,), 0.05)  # D is sure these are fake
d_loss, g_loss = both_losses(pred)
print('D-loss small, G-loss large:', round(float(d_loss), 4), round(float(g_loss), 4))
print('opposite directions:', bool(d_loss < g_loss))